### Week 6, Day 2

Before proceeding with our own MCP Server, let's just look at 2 popular marketplaces for what's out there:

https://glama.ai/mcp  
https://smithery.ai/servers 

We're about to create and use our own MCP Server!

It's pretty simple, but it's not super-simple. The excitement around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

backend/accounts.py

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

In [ ]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## Any guesses where this Account python module came from?!

I didn't write it!

In [ ]:
from backend.accounts import Account

In [ ]:
account = Account.get("Ed")
account.reset()
account

In [ ]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

In [ ]:
account.report()

In [ ]:
account.list_transactions()

### Now we write an MCP server and use it directly!

In [ ]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "-m", "backend.accounts_server"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [ ]:
mcp_tools

In [ ]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-5.4-mini"

In [ ]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to send a push notification, and then
            enjoy the outcome! There is a solution in backend/push_server.py if you need a hint.
            </span>
        </td>
    </tr>
</table>